# 06 — Scientific evaluation

**Purpose:** Validate prediction coverage, source hashes and label set before reporting macro-F1, accuracy and invalid-output rates.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


This notebook is deliberately strict. It rejects predictions for the wrong task, partial samples, wrong dataset versions, unknown example IDs and duplicate IDs. Invalid generated labels are counted as mistakes, not silently discarded.

In [ ]:
import pandas as pd
TASK='sib200'; LANG='ur'; SPLIT='validation'  # switch to 'test' only at final evaluation
paths=sorted((ROOT/'outputs/predictions').glob(f'*_{LANG}_{SPLIT}.csv'))
print('Found',len(paths),'prediction files')
for p in paths: print(p.name)

In [ ]:
results=[]
rows=load_task(TASK,LANG,SPLIT); labels=task_labels(TASK)
for p in paths:
    meta_path=p.with_suffix('.meta.json')
    if not meta_path.exists(): print('SKIP (no metadata):',p); continue
    meta=read_json(meta_path)
    if meta['task']!=TASK: print('SKIP other task:',p.name); continue
    try:
        validate_prediction_metadata(meta,TASK,LANG,SPLIT)
        if meta.get('subset'): raise ValueError('partial/smoke prediction set cannot be called full evaluation')
        predictions=read_csv(p)
        score=evaluate_predictions(rows,predictions,labels)
        compact={k:v for k,v in score.items() if k not in ['classification_report','confusion_matrix','confusion_labels']}
        compact.update({'run_id':meta['run_id'],'method':meta.get('method',meta['run_id']),
                        'task':TASK,'lang':LANG,'split':SPLIT,'file':str(p)})
        results.append(compact)
        write_json(ROOT/'outputs/metrics'/f'{p.stem}_metrics.json',dict(**compact, details=score, metadata=meta))
        print(p.name,'accuracy',round(score['accuracy'],4),'macro-F1',round(score['macro_f1'],4),
              'invalid',score['invalid_count'])
    except (ValueError,KeyError,FileNotFoundError) as e:
        print('REJECTED',p.name,'reason:',e)
if results:
    df=pd.DataFrame(results).sort_values('macro_f1',ascending=False)
    display(df[['run_id','split','count','accuracy','macro_f1','invalid_count']])
    df.to_csv(ROOT/'outputs/metrics'/f'{TASK}_{LANG}_{SPLIT}_leaderboard.csv',index=False)
else: print('No complete, verified predictions available yet.')